In [14]:
import os
import sqlite3
import shutil
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

BASE_DIR = "/content/fuel_inventory_intelligence"
RAW_DIR  = os.path.join(BASE_DIR, "data_raw")
OUT_DIR  = os.path.join(BASE_DIR, "data_output")
DB_PATH  = os.path.join(BASE_DIR, "fuel.db")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("✅ BASE_DIR:", BASE_DIR)
print("✅ RAW_DIR :", RAW_DIR)
print("✅ OUT_DIR :", OUT_DIR)
print("✅ DB_PATH :", DB_PATH)
np.random.seed(42)


✅ BASE_DIR: /content/fuel_inventory_intelligence
✅ RAW_DIR : /content/fuel_inventory_intelligence/data_raw
✅ OUT_DIR : /content/fuel_inventory_intelligence/data_output
✅ DB_PATH : /content/fuel_inventory_intelligence/fuel.db


In [15]:
def to_date_str(series: pd.Series) -> pd.Series:
    """
    Converts a datetime-like series to YYYY-MM-DD strings safely.
    NaT becomes empty string.
    """
    s = pd.to_datetime(series, errors="coerce")
    out = s.dt.strftime("%Y-%m-%d")
    return out.fillna("")


In [16]:
def create_sample_data(raw_dir: str):
    locations = pd.DataFrame({
        "location_id": ["LOC001", "LOC002", "LOC003"],
        "location_name": ["Station A", "Station B", "Station C"],
        "region": ["CT", "MA", "RI"],
        "tank_capacity_liters": [20000, 30000, 15000],
        "reorder_point_liters": [4000, 6000, 3000],
    })

    start = datetime(2026, 1, 1)
    days = 45
    dates = [start + timedelta(days=i) for i in range(days)]
    cons_rows = []
    for loc in locations["location_id"]:
        base = np.random.randint(900, 1600)
        for d in dates:
            cons_rows.append({
                "usage_date": d.strftime("%Y-%m-%d"),
                "location_id": loc,
                "fuel_type": "DIESEL",
                "consumed_liters": max(0, int(np.random.normal(base, 120)))
            })
    consumption = pd.DataFrame(cons_rows)
    del_rows = []
    order_id = 1000
    for loc in locations["location_id"]:
        for i in range(0, days, 10):
            order_date = start + timedelta(days=i)
            req = order_date + timedelta(days=2)
            actual = req + timedelta(days=int(np.random.choice([0, 0, 1, 2, 3])))
            del_rows.append({
                "order_id": f"ORD{order_id}",
                "location_id": loc,
                "fuel_type": "DIESEL",
                "order_date": order_date.strftime("%Y-%m-%d"),
                "requested_delivery_date": req.strftime("%Y-%m-%d"),
                "actual_delivery_date": actual.strftime("%Y-%m-%d"),
                "delivered_liters": int(np.random.choice([6000, 8000, 10000, 12000])),
                "carrier": str(np.random.choice(["CarrierX", "CarrierY", "CarrierZ"]))
            })
            order_id += 1
    deliveries = pd.DataFrame(del_rows)
    inv_rows = []
    for loc in locations["location_id"]:
        cap = float(locations.loc[locations["location_id"] == loc, "tank_capacity_liters"].iloc[0])
        onhand = cap * float(np.random.uniform(0.55, 0.8))

        dloc = deliveries[deliveries["location_id"] == loc].copy()
        dloc["actual_delivery_date"] = pd.to_datetime(dloc["actual_delivery_date"], errors="coerce")
        delivered_by_date = dloc.groupby(dloc["actual_delivery_date"].dt.date)["delivered_liters"].sum().to_dict()

        cloc = consumption[consumption["location_id"] == loc].copy()
        cloc["usage_date"] = pd.to_datetime(cloc["usage_date"], errors="coerce")
        consumed_by_date = cloc.groupby(cloc["usage_date"].dt.date)["consumed_liters"].sum().to_dict()

        for d in dates:
            dd = d.date()
            onhand -= float(consumed_by_date.get(dd, 0))
            onhand += float(delivered_by_date.get(dd, 0))
            onhand = max(0.0, min(cap, onhand))

            inv_rows.append({
                "snapshot_date": dd.strftime("%Y-%m-%d"),
                "location_id": loc,
                "fuel_type": "DIESEL",
                "onhand_liters": round(onhand, 2)
            })

    inventory = pd.DataFrame(inv_rows)
    locations.to_csv(os.path.join(raw_dir, "locations.csv"), index=False)
    consumption.to_csv(os.path.join(raw_dir, "consumption.csv"), index=False)
    deliveries.to_csv(os.path.join(raw_dir, "deliveries.csv"), index=False)
    inventory.to_csv(os.path.join(raw_dir, "inventory_snapshots.csv"), index=False)

    print("✅ Sample data created in:", raw_dir)

needed = ["locations.csv", "consumption.csv", "deliveries.csv", "inventory_snapshots.csv"]
if not all(os.path.exists(os.path.join(RAW_DIR, f)) for f in needed):
    create_sample_data(RAW_DIR)
else:
    print("✅ Found existing raw CSVs. Skipping sample generation.")


✅ Found existing raw CSVs. Skipping sample generation.


In [17]:
locations  = pd.read_csv(os.path.join(RAW_DIR, "locations.csv"))
consumption = pd.read_csv(os.path.join(RAW_DIR, "consumption.csv"))
deliveries  = pd.read_csv(os.path.join(RAW_DIR, "deliveries.csv"))
inventory   = pd.read_csv(os.path.join(RAW_DIR, "inventory_snapshots.csv"))

for df in [locations, consumption, deliveries, inventory]:
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()

for df in [consumption, deliveries, inventory]:
    if "fuel_type" in df.columns:
        df["fuel_type"] = df["fuel_type"].astype(str).str.upper().str.strip()

locations["tank_capacity_liters"] = pd.to_numeric(locations["tank_capacity_liters"], errors="coerce")
locations["reorder_point_liters"] = pd.to_numeric(locations["reorder_point_liters"], errors="coerce")
inventory["onhand_liters"] = pd.to_numeric(inventory["onhand_liters"], errors="coerce")
consumption["consumed_liters"] = pd.to_numeric(consumption["consumed_liters"], errors="coerce")
deliveries["delivered_liters"] = pd.to_numeric(deliveries["delivered_liters"], errors="coerce")

inventory["snapshot_date"] = pd.to_datetime(inventory["snapshot_date"], errors="coerce")
consumption["usage_date"]  = pd.to_datetime(consumption["usage_date"], errors="coerce")
for c in ["order_date", "requested_delivery_date", "actual_delivery_date"]:
    deliveries[c] = pd.to_datetime(deliveries[c], errors="coerce")

print("✅ Loaded shapes:")
print("locations  :", locations.shape)
print("consumption:", consumption.shape)
print("deliveries :", deliveries.shape)
print("inventory  :", inventory.shape)


✅ Loaded shapes:
locations  : (3, 5)
consumption: (135, 4)
deliveries : (15, 8)
inventory  : (135, 4)


In [18]:
def require_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing columns: {missing}")

def require_no_nulls(df, cols, name):
    bad = {c: int(df[c].isna().sum()) for c in cols}
    bad = {k: v for k, v in bad.items() if v > 0}
    if bad:
        raise ValueError(f"{name}: nulls found: {bad}")

def require_non_negative(df, cols, name):
    for c in cols:
        if (df[c] < 0).any():
            raise ValueError(f"{name}: negative values in {c}")

def require_unique(df, key_cols, name):
    if df.duplicated(key_cols).any():
        sample = df[df.duplicated(key_cols, keep=False)][key_cols].head(10)
        raise ValueError(f"{name}: duplicate keys {key_cols}. Sample:\n{sample}")

require_cols(locations, ["location_id","location_name","tank_capacity_liters","reorder_point_liters"], "locations")
require_no_nulls(locations, ["location_id","location_name","tank_capacity_liters","reorder_point_liters"], "locations")
require_non_negative(locations, ["tank_capacity_liters","reorder_point_liters"], "locations")
require_unique(locations, ["location_id"], "locations")
if (locations["reorder_point_liters"] > locations["tank_capacity_liters"]).any():
    raise ValueError("locations: reorder_point_liters cannot exceed tank_capacity_liters")

require_cols(inventory, ["snapshot_date","location_id","fuel_type","onhand_liters"], "inventory")
require_no_nulls(inventory, ["snapshot_date","location_id","fuel_type","onhand_liters"], "inventory")
require_non_negative(inventory, ["onhand_liters"], "inventory")
require_unique(inventory, ["snapshot_date","location_id","fuel_type"], "inventory")

require_cols(consumption, ["usage_date","location_id","fuel_type","consumed_liters"], "consumption")
require_no_nulls(consumption, ["usage_date","location_id","fuel_type","consumed_liters"], "consumption")
require_non_negative(consumption, ["consumed_liters"], "consumption")
require_unique(consumption, ["usage_date","location_id","fuel_type"], "consumption")

# deliveries
require_cols(deliveries, ["order_id","location_id","fuel_type","order_date","delivered_liters"], "deliveries")
require_no_nulls(deliveries, ["order_id","location_id","fuel_type","order_date","delivered_liters"], "deliveries")
require_non_negative(deliveries, ["delivered_liters"], "deliveries")
require_unique(deliveries, ["order_id"], "deliveries")

print("✅ Validation passed")


✅ Validation passed


In [19]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS dim_location;
DROP TABLE IF EXISTS fact_inventory_snapshot;
DROP TABLE IF EXISTS fact_delivery;
DROP TABLE IF EXISTS fact_consumption;

CREATE TABLE dim_location (
  location_id TEXT PRIMARY KEY,
  location_name TEXT NOT NULL,
  region TEXT,
  tank_capacity_liters REAL NOT NULL,
  reorder_point_liters REAL NOT NULL
);

CREATE TABLE fact_inventory_snapshot (
  snapshot_date TEXT NOT NULL,
  location_id TEXT NOT NULL,
  fuel_type TEXT NOT NULL,
  onhand_liters REAL NOT NULL
);

CREATE TABLE fact_delivery (
  order_id TEXT PRIMARY KEY,
  location_id TEXT NOT NULL,
  fuel_type TEXT NOT NULL,
  order_date TEXT NOT NULL,
  requested_delivery_date TEXT,
  actual_delivery_date TEXT,
  delivered_liters REAL NOT NULL,
  carrier TEXT
);

CREATE TABLE fact_consumption (
  usage_date TEXT NOT NULL,
  location_id TEXT NOT NULL,
  fuel_type TEXT NOT NULL,
  consumed_liters REAL NOT NULL
);
""")
conn.commit()

locations.to_sql("dim_location", conn, if_exists="append", index=False)

inventory_db = inventory.copy()
inventory_db["snapshot_date"] = to_date_str(inventory_db["snapshot_date"])
inventory_db.to_sql("fact_inventory_snapshot", conn, if_exists="append", index=False)

cons_db = consumption.copy()
cons_db["usage_date"] = to_date_str(cons_db["usage_date"])
cons_db.to_sql("fact_consumption", conn, if_exists="append", index=False)

del_db = deliveries.copy()
del_db["order_date"] = to_date_str(del_db["order_date"])
del_db["requested_delivery_date"] = to_date_str(del_db["requested_delivery_date"])
del_db["actual_delivery_date"] = to_date_str(del_db["actual_delivery_date"])
del_db.to_sql("fact_delivery", conn, if_exists="append", index=False)

print("✅ Loaded into SQLite:", DB_PATH)


✅ Loaded into SQLite: /content/fuel_inventory_intelligence/fuel.db


In [20]:
cons_sorted = consumption.sort_values(["location_id","fuel_type","usage_date"]).copy()
cons_sorted["avg_daily_consumption_7d"] = (
    cons_sorted.groupby(["location_id","fuel_type"])["consumed_liters"]
    .rolling(7, min_periods=1).mean()
    .reset_index(level=[0,1], drop=True)
)

inv_sorted = inventory.sort_values(["location_id","fuel_type","snapshot_date"]).copy()

days_supply = inv_sorted.merge(
    cons_sorted.rename(columns={"usage_date":"snapshot_date"}),
    on=["location_id","fuel_type","snapshot_date"],
    how="left"
)

days_supply["avg_daily_consumption_7d"] = days_supply["avg_daily_consumption_7d"].fillna(0.0)
days_supply["days_of_supply"] = np.where(
    days_supply["avg_daily_consumption_7d"] > 0,
    days_supply["onhand_liters"] / days_supply["avg_daily_consumption_7d"],
    np.nan
)

end = max(consumption["usage_date"].max(), inventory["snapshot_date"].max())
start = end - pd.Timedelta(days=29)

cons_30 = consumption[(consumption["usage_date"] >= start) & (consumption["usage_date"] <= end)]
inv_30 = inventory[(inventory["snapshot_date"] >= start) & (inventory["snapshot_date"] <= end)]

cons_sum = cons_30.groupby(["location_id","fuel_type"], as_index=False)["consumed_liters"].sum()
inv_avg = inv_30.groupby(["location_id","fuel_type"], as_index=False)["onhand_liters"].mean().rename(columns={"onhand_liters":"avg_onhand_liters"})

turnover_30d = cons_sum.merge(inv_avg, on=["location_id","fuel_type"], how="left")
turnover_30d["inventory_turnover"] = np.where(
    turnover_30d["avg_onhand_liters"] > 0,
    turnover_30d["consumed_liters"] / turnover_30d["avg_onhand_liters"],
    np.nan
)
turnover_30d["period_start"] = start.strftime("%Y-%m-%d")
turnover_30d["period_end"] = end.strftime("%Y-%m-%d")

d = deliveries.copy()
d["lead_time_days"] = (d["actual_delivery_date"] - d["order_date"]).dt.days
d["schedule_slip_days"] = (d["actual_delivery_date"] - d["requested_delivery_date"]).dt.days

replen_eff = d.groupby(["location_id","fuel_type"], as_index=False).agg(
    avg_lead_time_days=("lead_time_days","mean"),
    p90_lead_time_days=("lead_time_days", lambda x: np.nanpercentile(x.dropna(), 90) if x.dropna().size else np.nan),
    avg_schedule_slip_days=("schedule_slip_days","mean"),
    on_time_rate=("schedule_slip_days", lambda x: float((x.fillna(0) <= 0).mean()))
)

print("✅ KPI tables computed:")
print("days_supply  :", days_supply.shape)
print("turnover_30d :", turnover_30d.shape)
print("replen_eff   :", replen_eff.shape)


✅ KPI tables computed:
days_supply  : (135, 7)
turnover_30d : (3, 7)
replen_eff   : (3, 6)


In [21]:
STOCKOUT_DAYS_THRESHOLD = 3
OVERSTOCK_DAYS_THRESHOLD = 30

days_supply["stockout_risk_flag"] = (
    days_supply["days_of_supply"].notna() &
    (days_supply["days_of_supply"] <= STOCKOUT_DAYS_THRESHOLD)
)

days_supply["overstock_flag"] = (
    days_supply["days_of_supply"].notna() &
    (days_supply["days_of_supply"] >= OVERSTOCK_DAYS_THRESHOLD)
)

print("✅ Risk flags added")


✅ Risk flags added


In [22]:
c_agg = consumption.groupby(["location_id","fuel_type"], as_index=False).agg(
    avg_daily_use=("consumed_liters","mean"),
    std_daily_use=("consumed_liters","std"),
)
c_agg["consumption_cv"] = np.where(
    c_agg["avg_daily_use"] > 0,
    c_agg["std_daily_use"] / c_agg["avg_daily_use"],
    np.nan
)

d_agg = d.groupby(["location_id","fuel_type"], as_index=False).agg(
    avg_lead_time=("lead_time_days","mean"),
    std_lead_time=("lead_time_days","std"),
)

drivers = c_agg.merge(d_agg, on=["location_id","fuel_type"], how="outer")
drivers["variability_score"] = (
    drivers["consumption_cv"].fillna(0) * 0.6 +
    (drivers["std_lead_time"].fillna(0) / 10.0) * 0.4
)

print("✅ Drivers computed:", drivers.shape)


✅ Drivers computed: (3, 8)


In [23]:
loc_small = locations[["location_id","location_name","region","tank_capacity_liters","reorder_point_liters"]].copy()

days_supply_out = days_supply.copy()
days_supply_out["snapshot_date"] = to_date_str(days_supply_out["snapshot_date"])
days_supply_out = days_supply_out.merge(loc_small, on="location_id", how="left")

turnover_out = turnover_30d.merge(loc_small, on="location_id", how="left")
replen_out = replen_eff.merge(loc_small, on="location_id", how="left")
drivers_out = drivers.merge(loc_small, on="location_id", how="left")

tables = {
    "dim_location": locations,
    "fact_inventory_snapshot": inventory.assign(snapshot_date=to_date_str(inventory["snapshot_date"])),
    "fact_consumption": consumption.assign(usage_date=to_date_str(consumption["usage_date"])),
    "fact_delivery": deliveries.assign(
        order_date=to_date_str(deliveries["order_date"]),
        requested_delivery_date=to_date_str(deliveries["requested_delivery_date"]),
        actual_delivery_date=to_date_str(deliveries["actual_delivery_date"]),
    ),
    "kpi_days_of_supply": days_supply_out,
    "kpi_inventory_turnover_30d": turnover_out,
    "kpi_replenishment_efficiency": replen_out,
    "kpi_supply_variability_drivers": drivers_out,
}

for name, df in tables.items():
    out_path = os.path.join(OUT_DIR, f"{name}.csv")
    df.to_csv(out_path, index=False)

print("✅ Exported CSVs to:", OUT_DIR)
print("Files:", sorted(os.listdir(OUT_DIR))[:10], "...")


✅ Exported CSVs to: /content/fuel_inventory_intelligence/data_output
Files: ['dim_location.csv', 'fact_consumption.csv', 'fact_delivery.csv', 'fact_inventory_snapshot.csv', 'kpi_days_of_supply.csv', 'kpi_inventory_turnover_30d.csv', 'kpi_replenishment_efficiency.csv', 'kpi_supply_variability_drivers.csv'] ...


In [24]:
display(days_supply_out.head())
display(turnover_out.head())
display(replen_out.head())
display(drivers_out.head())


,snapshot_date,location_id,fuel_type,onhand_liters,consumed_liters,avg_daily_consumption_7d,days_of_supply,stockout_risk_flag,overstock_flag,location_name,region,tank_capacity_liters,reorder_point_liters
0,2026-01-01,LOC001,DIESEL,11926.41,935,935.000000,12.755519,False,False,Station A,CT,20000,4000
1,2026-01-02,LOC001,DIESEL,10863.41,1063,999.000000,10.874284,False,False,Station A,CT,20000,4000
2,2026-01-03,LOC001,DIESEL,9805.41,1058,1018.666667,9.625730,False,False,Station A,CT,20000,4000
3,2026-01-04,LOC001,DIESEL,8639.41,1166,1055.500000,8.185135,False,False,Station A,CT,20000,4000
4,2026-01-05,LOC001,DIESEL,13748.41,891,1022.600000,13.444563,False,False,Station A,CT,20000,4000


,location_id,fuel_type,consumed_liters,avg_onhand_liters,inventory_turnover,period_start,period_end,location_name,region,tank_capacity_liters,reorder_point_liters
0,LOC001,DIESEL,30155,13964.91,2.159341,2026-01-16,2026-02-14,Station A,CT,20000,4000
1,LOC002,DIESEL,36843,16065.80,2.293256,2026-01-16,2026-02-14,Station B,MA,30000,6000
2,LOC003,DIESEL,44346,3947.40,11.234230,2026-01-16,2026-02-14,Station C,RI,15000,3000


,location_id,fuel_type,avg_lead_time_days,p90_lead_time_days,avg_schedule_slip_days,on_time_rate,location_name,region,tank_capacity_liters,reorder_point_liters
0,LOC001,DIESEL,2.6,3.6,0.6,0.6,Station A,CT,20000,4000
1,LOC002,DIESEL,3.0,4.0,1.0,0.4,Station B,MA,30000,6000
2,LOC003,DIESEL,2.6,3.6,0.6,0.6,Station C,RI,15000,3000


,location_id,fuel_type,avg_daily_use,std_daily_use,consumption_cv,avg_lead_time,std_lead_time,variability_score,location_name,region,tank_capacity_liters,reorder_point_liters
0,LOC001,DIESEL,995.133333,104.170882,0.104680,2.6,0.894427,0.098585,Station A,CT,20000,4000
1,LOC002,DIESEL,1227.511111,134.682729,0.109720,3.0,1.000000,0.105832,Station B,MA,30000,6000
2,LOC003,DIESEL,1471.800000,114.381459,0.077715,2.6,0.894427,0.082406,Station C,RI,15000,3000


In [25]:
zip_path = shutil.make_archive(os.path.join(BASE_DIR, "powerbi_outputs"), "zip", OUT_DIR)
print("✅ Zipped outputs:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download skipped (not in Colab). Error:", str(e))


✅ Zipped outputs: /content/fuel_inventory_intelligence/powerbi_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
try:
    conn.close()
    print("✅ SQLite connection closed")
except Exception as e:
    print("DB close skipped:", str(e))


✅ SQLite connection closed
